# معمل التجربة: تحديث وإضافة مستندات جديدة لقاعدة المتجهات

في الشغل الحقيقي، قاعدة بيانات الـ RAG مش ثابتة، كل يوم بتنزل وثائق جديدة ومستندات بتتعدل.
في المعمل ده هنجرب:
1. إنشاء قاعدة بيانات ChromaDB بسيطة.
2. إضافة مستند جديد إليها بعد إنشائها (`add_documents`).
3. التأكد من إن المستند الجديد بقى متاح فورا في نتائج البحث والاسترجاع.

### 1. تجهيز قاعدة البيانات الأولية
يلا نضيف مستندين في البداية:

In [1]:
import tempfile
import shutil
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

load_dotenv()
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
temp_dir = tempfile.mkdtemp()

initial_docs = [
    Document(page_content="التعلم الخاضع للإشراف (Supervised Learning) يعتمد على بيانات معنونة للتدريب."),
    Document(page_content="التعلم غير الخاضع للإشراف (Unsupervised Learning) يكتشف الأنماط بدون تسميات مسبقة."),
]

vector_store = Chroma.from_documents(initial_docs, embeddings, persist_directory=temp_dir)
print(f"عدد المستندات في البداية: {vector_store._collection.count()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

عدد المستندات في البداية: 2


### 2. إضافة مستند جديد في وقت التشغيل
دلوقتي هنضيف مستند عن التعلم المعزز (Reinforcement Learning):

In [2]:
new_doc = Document(
    page_content="التعلم المعزز (Reinforcement Learning) يدرب الوكيل الذكي على اتخاذ قرارات متتالية للحصول على أكبر مكافأة ممكنة في بيئته."
)

added_ids = vector_store.add_documents([new_doc])
print(f"تمت إضافة المستند بنجاح! المعرف: {added_ids}")
print(f"العدد الإجمالي الجديد: {vector_store._collection.count()}")

تمت إضافة المستند بنجاح! المعرف: ['27972eef-2f5d-49d6-8fd0-db1fb188c217']
العدد الإجمالي الجديد: 3


### 3. تجربة البحث في المعرفة المحدثة
تعال نبحث عن موضوع المستند الجديد:

In [3]:
results = vector_store.similarity_search("كيف يتعلم الوكيل الذكي عبر المكافآت؟", k=1)
print("النتيجة المسترجعة:")
print(results[0].page_content)

# تنظيف
shutil.rmtree(temp_dir, ignore_errors=True)

النتيجة المسترجعة:
التعلم غير الخاضع للإشراف (Unsupervised Learning) يكتشف الأنماط بدون تسميات مسبقة.


### 4. جرب بنفسك!
جرب تضيف نصوص تانية عن اهتماماتك وافحص عدد المستندات بعد الإضافة.